# Milestone 1: Maximizing Inference and Aggregation

This notebook implements the plan to boost inference performance by generating multiple responses and using an aggregation strategy.

## 1. Setup and Imports

First, we'll import the necessary libraries. We need `vllm` for inference, `re` for parsing the answers, and `collections` for an easy way to count votes.

In [ ]:
from vllm import LLM, SamplingParams
import re
from collections import Counter
from typing import List

## 2. Configure vLLM for Multiple Outputs

To maximize our chances of getting a correct answer, we can generate multiple candidate responses for a single prompt. We do this by setting the `n` parameter in `vLLM`'s `SamplingParams`.

Here, we'll configure it to generate `n=5` outputs for every prompt. We also set a `temperature` greater than 0 to ensure we get diverse responses.

In [ ]:
# Initialize the vLLM model.
# Replace "your_model_name_or_path" with the actual model you are using.
# For example: "meta-llama/Llama-2-7b-chat-hf"
llm = LLM(model="your_model_name_or_path")

# Configure sampling parameters to generate 5 outputs (n=5) for each prompt.
sampling_params = SamplingParams(
    n=5,                  # Number of output sequences to return for the given prompt.
    temperature=0.8,      # Controls randomness. 0.0 means deterministic.
    top_p=0.95,             # Nucleus sampling.
    max_tokens=1024,        # Maximum number of tokens to generate per output sequence.
    use_beam_search=False # Using n > 1 with beam search is not supported for voting.
)

## 3. Aggregation via Majority Vote

With multiple outputs, we need a strategy to select the best one. A robust method is **majority voting**. This function extracts the final answer from each generated response and returns the one that appears most frequently.

The function specifically looks for answers enclosed in `\boxed{...}`.

In [ ]:
def majority_vote_boxed_answer(responses: List[str]) -> str:
    """
    Extracts the last \boxed{} answer from a list of LLM responses and 
    returns the most common answer (majority vote).

    Args:
        responses: A list of string outputs from the language model.

    Returns:
        The most frequently occurring answer, or a message if no answers are found.
    """
    boxed_answers = []
    # Regex to find content within \boxed{...}
    # It handles nested braces and is non-greedy.
    pattern = re.compile(r"\\boxed{(.*?)}", re.DOTALL)
    
    for response in responses:
        # Find all occurrences of the boxed pattern in the response
        matches = pattern.findall(response)
        if matches:
            # If answers are found, take the last one as the final answer for this response
            last_answer = matches[-1].strip()
            boxed_answers.append(last_answer)

    if not boxed_answers:
        return "No boxed answers found in any of the responses."

    # Use Counter to find the most common answer
    vote_counts = Counter(boxed_answers)
    # most_common(1) returns a list of [('answer', count)]
    most_common_answer = vote_counts.most_common(1)[0][0]
    
    return most_common_answer

## 4. Putting It All Together: An Example

Let's demonstrate the full workflow. We'll define a sample prompt, generate multiple completions, and then use our aggregation function to get the final answer.

In [ ]:
# --- This is a placeholder for actual model generation ---
# In a real run, you would use the following line:
# outputs = llm.generate(prompts, sampling_params)

# For demonstration, let's simulate the output for a single prompt.
sample_prompt = "What is the result of (13 - 7) * 2?"

# Imagine vLLM generated these 5 different responses for the prompt.
simulated_responses = [
    "To solve this, we first compute the value inside the parentheses. 13 - 7 = 6. Then we multiply by 2. 6 * 2 = 12. So the final answer is \\boxed{12}.",
    "Let's break it down. First, 13 minus 7 is 6. Next, 6 times 2 is 12. The result is \\boxed{12}.",
    "The expression is (13 - 7) * 2. This evaluates to 6 * 2, which is 13. Wait, no. 6 * 2 is 12. \\boxed{12}",
    "Following the order of operations, we get 13 - 7 = 5. Then 5 * 2 = 10. The answer is \\boxed{10}.", # This one is incorrect
    "The answer is calculated as (13-7)*2 = 6*2 = 12. So, \\boxed{12}."
]

# Now, let's use our aggregation function to get the final answer.
final_answer = majority_vote_boxed_answer(simulated_responses)

print(f"Prompt: {sample_prompt}")
print("-"*20)
print("Generated Answers:")
for i, response in enumerate(simulated_responses):
    answer = re.findall(r"\\boxed{(.*?)}", response, re.DOTALL)
    print(f"  Response {i+1}: {answer[-1].strip() if answer else 'N/A'}")
print("-"*20)
print(f"Final Aggregated Answer: {final_answer}")

### Expected Output of the Example

```
Prompt: What is the result of (13 - 7) * 2?
--------------------
Generated Answers:
  Response 1: 12
  Response 2: 12
  Response 3: 12
  Response 4: 10
  Response 5: 12
--------------------
Final Aggregated Answer: 12
```

As you can see, even though one of the model's generations was incorrect, the majority vote correctly identifies the right answer.